# Notebook 03 — GTFS stop density per commune, Hauts-de-France

Third lens for the analysis: public transport access, measured as the count of GTFS transit stops per commune, crossed with the 2026 municipal election results.

**Source:** transport.data.gouv.fr API (`/api/gtfs-stops`) — real-time GTFS aggregation by bounding box.

**Output:** `data/processed/transport-hdf-elections.json` — 88 HDF communes with columns: `code_commune`, `nom_commune`, `abstention_rate`, `winning_bloc`, `n_stops`.

## 1. Fetch GTFS stops — bounding box tiling strategy

The full HDF bounding box is too large for a single API call: the endpoint caps results at 20,000 stops. To stay under this limit, the region is divided into a 4×5 grid of smaller tiles (roughly 0.5° lat × 0.7° lon each), covering 49.0–51.1°N and 1.4–4.2°E.

Each tile is fetched separately. Stops are deduplicated across tiles using a composite key of `stop_id` + `dataset_id`, since the same physical stop can appear in multiple overlapping GTFS datasets. Result: **64,575 unique stops** across HDF.

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape
import time

# HDF bounding box extremes
SOUTH, NORTH, WEST, EAST = 49.0, 51.1, 1.4, 4.2

# Grid breakpoints — 5 lat bands × 4 lon bands = 20 tiles
# Narrower top band (51.0–51.1) because HDF barely extends that far north
lat_steps = [49.0, 49.5, 50.0, 50.5, 51.0, 51.1]
lon_steps = [1.4, 2.1, 2.8, 3.5, 4.2]

BASE_URL = "https://transport.data.gouv.fr/api/gtfs-stops"

all_features = []   # accumulates all unique stop GeoJSON features
seen_ids = set()    # deduplication set — composite key: stop_id + dataset_id
errors = []         # tiles that failed; inspected after the loop

for i in range(len(lat_steps) - 1):
    for j in range(len(lon_steps) - 1):
        s = lat_steps[i]
        n = lat_steps[i+1]
        w = lon_steps[j]
        e = lon_steps[j+1]

        params = {'south': s, 'north': n, 'west': w, 'east': e}
        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
            data = r.json()

            # API returns {"error": "..."} for bad requests
            if isinstance(data, dict) and 'error' in data:
                errors.append(f"Tile ({s},{w})-({n},{e}): {data['error']}")
                continue

            features = data.get('features', [])
            new = 0
            for f in features:
                sid = f['properties'].get('stop_id', '')
                # Composite key prevents duplicates from overlapping datasets
                key = f"{sid}_{f['properties'].get('dataset_id', '')}"
                if key not in seen_ids:
                    seen_ids.add(key)
                    all_features.append(f)
                    new += 1
            print(f"Tile ({s:.1f},{w:.1f})-({n:.1f},{e:.1f}): {len(features)} stops, {new} new")
        except Exception as ex:
            errors.append(f"Tile ({s},{w})-({n},{e}): {ex}")

        time.sleep(0.3)  # polite rate limiting — avoid hammering the API

print(f"\nTotal unique stops: {len(all_features)}")
if errors:
    print("Errors:", errors)

## 2. Load commune boundaries for HDF

Fetch commune polygons for all five HDF departments (Nord 59, Pas-de-Calais 62, Aisne 02, Oise 60, Somme 80) from the geo.api.gouv.fr API. Returns GeoJSON with full polygon contours — no file download required.

Total: **3,782 communes** across HDF, which will be used as the spatial target for assigning stops.

In [ ]:
# Fetch commune boundaries for HDF departments via geo.api.gouv.fr
hdf_depts = ['59', '62', '02', '60', '80']
communes_list = []

for dept in hdf_depts:
    url = f"https://geo.api.gouv.fr/departements/{dept}/communes"
    # Request full polygon contour in GeoJSON; exclude arrondissements
    params = {'geometry': 'contour', 'format': 'geojson', 'type': 'commune-actuelle'}
    r = requests.get(url, params=params, timeout=30)
    gdf = gpd.read_file(r.text)
    communes_list.append(gdf)
    print(f"Dept {dept}: {len(gdf)} communes")

# Concatenate all five departments into one GeoDataFrame
communes_hdf = pd.concat(communes_list, ignore_index=True)
print(f"\nTotal communes HDF: {len(communes_hdf)}")
print(communes_hdf.columns.tolist())
print(communes_hdf.head(3))

## 3. Spatial join — assign each stop to a commune

Each stop is a lat/lon point. Using a GeoPandas spatial join (`within` predicate), each stop is matched to the commune polygon it falls inside.

About 10,000 stops (~15%) fall outside any commune polygon — these are stops near borders or in neighbouring regions that were captured by the bounding box tiles. They are left unmatched (`NaN`) and dropped in the aggregation step.

In [ ]:
# Build a GeoDataFrame from the raw GeoJSON features collected in the tiling loop
stop_records = []
for f in all_features:
    coords = f['geometry']['coordinates']  # GeoJSON order: [lon, lat]
    props = f['properties']
    stop_records.append({
        'stop_id': props.get('stop_id'),
        'stop_name': props.get('stop_name'),
        'dataset_title': props.get('dataset_title'),
        'location_type': props.get('location_type'),  # 0 = stop/platform, 1 = parent station
        'geometry': Point(coords[0], coords[1])
    })

stops_gdf = gpd.GeoDataFrame(stop_records, crs='EPSG:4326')
print(f"Stops GeoDataFrame: {len(stops_gdf)} rows")

# Align CRS before spatial join
communes_hdf = communes_hdf.to_crs('EPSG:4326')

# Left join: stops that don't fall in any commune get NaN for code/nom
joined = gpd.sjoin(stops_gdf, communes_hdf[['code', 'nom', 'geometry']],
                   how='left', predicate='within')

print(f"Stops matched to commune: {joined['code'].notna().sum()} / {len(joined)}")
print(joined.head(3))

## 4. Aggregate stops per commune and join election data

Count stops per commune (physical stops only, `location_type == 0`), then join against the election dataset filtered to the 88 HDF communes that have valid election results.

Communes with no stops in the GTFS data get `n_stops = 0` (all 88 HDF election communes have at least some coverage).

In [ ]:
# Count stops per commune — filter to location_type==0 (individual stop/platform)
# location_type 1 = parent station (the hub), which would double-count its child stops
stops_per_commune = (
    joined[joined['location_type'] == 0]
    .groupby('code')
    .agg(n_stops=('stop_id', 'count'))
    .reset_index()
    .rename(columns={'code': 'code_commune'})
)

print(f"Communes with stop data: {len(stops_per_commune)}")
print(stops_per_commune.sort_values('n_stops', ascending=False).head(10))

# --- Rebuild election data for HDF ---
# Load pre-processed commune-level parquet produced in notebook 01/02
df = pd.read_parquet('../data/raw/commune.parquet')

# Parse abstention rate from French percentage string ("45,3%") to float
df['abstention_rate'] = df['% Abstentions'].str.replace('%', '').str.replace(',', '.').astype(float)

# Identify winning list: whichever of the 5 candidate columns has the highest vote share
voix_cols = [f'% Voix/exprimés {i}' for i in range(1, 6)]
for col in voix_cols:
    df[col] = pd.to_numeric(df[col].str.replace('%', '').str.replace(',', '.'), errors='coerce')

# idxmax returns the column name of the max value — extract the list number from it
winning_col = df[voix_cols].idxmax(axis=1)
winning_num = winning_col.str.split(' ').str[-1]  # e.g. "% Voix/exprimés 3" → "3"

# Look up the nuance code for the winning list number in each row
df['winning_nuance'] = [row[f'Nuance liste {num}'] for (_, row), num in zip(df.iterrows(), winning_num)]

# Join nuances.csv to resolve nuance code → political bloc
nuances = pd.read_csv('../data/raw/nuances.csv', sep=';')
nuances_liste = nuances[nuances['type_nuance'] == 'liste']  # exclude candidate-level nuances
df = df.merge(
    nuances_liste[['libelle_nuance', 'bloc']].rename(columns={'bloc': 'winning_bloc'}),
    left_on='winning_nuance', right_on='libelle_nuance', how='left'
)

# Drop communes where bloc couldn't be resolved (unrecognised nuance codes)
df_clean = df[df['winning_bloc'].notna()].copy()

# Filter to HDF departments — extract dept code from 5-digit commune code
hdf_depts_codes = ['59', '62', '02', '60', '80']
df_clean['dept'] = df_clean['Code commune'].astype(str).str.zfill(5).str[:2]
df_hdf = df_clean[df_clean['dept'].isin(hdf_depts_codes)].copy()
print(f"HDF election communes: {len(df_hdf)}")

# Join stop counts — communes with no stops get 0
df_hdf_transport = df_hdf.merge(
    stops_per_commune,
    left_on='Code commune',
    right_on='code_commune',
    how='left'
)
df_hdf_transport['n_stops'] = df_hdf_transport['n_stops'].fillna(0).astype(int)

print(f"\nHDF communes with stops: {(df_hdf_transport['n_stops'] > 0).sum()} / {len(df_hdf_transport)}")
print("\nStops vs abstention by bloc:")
print(df_hdf_transport.groupby('winning_bloc').agg(
    communes=('Code commune', 'count'),
    mean_stops=('n_stops', 'mean'),
    mean_abstention=('abstention_rate', 'mean')
).sort_values('mean_stops', ascending=False))

## 5. Scatter plot — transport access vs abstention rate, by winning bloc

One dot per commune. X-axis: number of GTFS stops (transport access proxy). Y-axis: abstention rate. Colour: winning political bloc.

This single chart answers both research questions simultaneously: the y-axis shows whether transport access correlates with turnout; the colour shows whether it correlates with political outcome.

In [ ]:
import plotly.express as px
import json, os

# Canonical bloc colour palette (Le Monde conventions)
bloc_colors = {
    'EXG': '#cc0000',
    'GAU': '#e75480',
    'CENT': '#ffcc00',
    'DTE': '#0055a4',
    'EXD': '#1a1a2e',
    'DIV': '#aaaaaa',
}

fig = px.scatter(
    df_hdf_transport,
    x='n_stops',
    y='abstention_rate',
    color='winning_bloc',
    color_discrete_map=bloc_colors,
    hover_name='Libellé commune',
    hover_data={'n_stops': True, 'abstention_rate': ':.1f', 'winning_bloc': True},
    labels={
        'n_stops': "Nombre d'arrêts de transport en commun",
        'abstention_rate': "Taux d'abstention (%)",
        'winning_bloc': 'Bloc vainqueur',
    },
    title='Transport en commun et abstention — Hauts-de-France, Municipales 2026',
    opacity=0.75,
)
fig.update_traces(marker=dict(size=8))
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(showgrid=True, gridcolor='#eeeeee'),
)
fig.show()

# --- Export ---
os.makedirs('../data/processed', exist_ok=True)

# Keep only the columns needed for the web viz
export_cols = ['Code commune', 'Libellé commune', 'abstention_rate', 'winning_bloc', 'n_stops']
df_hdf_transport[export_cols].rename(columns={
    'Code commune': 'code_commune',
    'Libellé commune': 'nom_commune'
}).to_json(
    '../data/processed/transport-hdf-elections.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("Exported transport-hdf-elections.json")